# Stage 6A: deterministic daily planner

This notebook shows how user ratings, retrieved records, configured time bounds, and deterministic rules become one-day plans. The weights and timings are prototype product settings, not universal scientific values.

In [ ]:
import json
from pathlib import Path

from src.daily_planner import DailyPlanner
from src.planner_config import default_planner_config
from src.planner_models import DailyPlanRequest
from src.retriever import RetrievalService
from src.subject_scoring import score_subjects

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
demo_inputs = json.loads((PROJECT_ROOT / 'data' / 'processed' / 'planner_demo_inputs.json').read_text(encoding='utf-8'))
config = default_planner_config(PROJECT_ROOT)
print(config.planner_version, config.planning_weights)

## Score calculation

Knowledge gap is `6 - current understanding`. The four 1-to-5 values are combined with the configured weights.

In [ ]:
school_request = DailyPlanRequest.from_dict(demo_inputs[0]['request'])
school_scores = score_subjects(school_request.subjects, config)
for item in school_scores:
    print(item.subject.name, {'score': item.score, 'knowledge_gap': item.knowledge_gap, 'demand': item.cognitive_demand})

## School and university examples

Both examples use the frozen retriever. Only record IDs and planner-safe method fields flow into the final sessions.

In [ ]:
retrieval_service = RetrievalService.from_frozen_config(PROJECT_ROOT, device='cpu')
planner = DailyPlanner(PROJECT_ROOT, retrieval_service=retrieval_service)
school_plan = planner.plan(demo_inputs[0]['request'])
university_plan = planner.plan(demo_inputs[1]['request'])
print('school:', school_plan.plan_id)
print('university:', university_plan.plan_id)

## Retrieved knowledge and time allocation

Canonical names show alias resolution. Record IDs keep the plan traceable without exposing internal retrieval text.

In [ ]:
for item in university_plan.scheduled_subjects:
    print(item.subject, '->', item.canonical_subject, item.allocated_study_minutes, 'minutes', list(item.retrieved_record_ids))

## Final ordered plan

Breaks count inside the available window. Lower-demand work separates high-demand sessions when possible.

In [ ]:
for item in university_plan.sessions:
    label = item.subject or 'break'
    print(item.order, item.start_time, label, item.topic, item.duration_minutes)
assert university_plan.total_planned_minutes <= university_plan.total_available_minutes
assert university_plan.total_study_minutes + university_plan.total_break_minutes == university_plan.total_planned_minutes
print('constraint confirmed:', university_plan.total_planned_minutes, '<=', university_plan.total_available_minutes)

In [ ]:
repeated = planner.plan(demo_inputs[1]['request'])
assert repeated.to_dict() == university_plan.to_dict()
retrieval_service.close()
print('deterministic repeated output confirmed')